# RoleLens 핵심 기술 프로토타입

이 노트북은 운영 진입점이 아니라 `RunnableBranch → AnalysisPlan → PostgreSQL → DataFrame → 차트 → 근거 요약`의 기술 검증용이다. 비밀값은 환경변수에서만 읽는다.

In [1]:
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import pandas as pd
from langchain_core.runnables import RunnableBranch, RunnableLambda
from pydantic import BaseModel
from sqlalchemy import text

from rolelens.config import get_settings
from rolelens.db.engine import create_readonly_engine

settings = get_settings()
PROJECT_ROOT = Path.cwd() if (Path.cwd() / 'sql').exists() else Path.cwd().parent
print({'provider': settings.model_provider, 'model': settings.model_id, 'api_key_set': settings.model_api_key is not None})

{'provider': 'openai', 'model': 'gpt-4o-mini', 'api_key_set': True}


## 1. 동일 출력 계약을 갖는 페르소나 분기

In [2]:
class PrototypeAnalysisPlan(BaseModel):
    persona: Literal['planner', 'marketer', 'pm']
    metric: str
    dimensions: list[str]
    comparison: str | None
    chart_hint: str

def plan(persona: str, dimensions: list[str], comparison: str, chart_hint: str):
    return RunnableLambda(lambda x: PrototypeAnalysisPlan(
        persona=persona, metric='revenue', dimensions=dimensions,
        comparison=comparison, chart_hint=chart_hint))

analysis_branch = RunnableBranch(
    (lambda x: x['persona'] == 'planner', plan('planner', ['month'], 'target', 'grouped_bar')),
    (lambda x: x['persona'] == 'marketer', plan('marketer', ['category'], 'previous_period', 'grouped_bar')),
    (lambda x: x['persona'] == 'pm', plan('pm', ['device'], 'funnel_stage', 'bar')),
    RunnableLambda(lambda x: (_ for _ in ()).throw(ValueError('unsupported persona'))),
)
prototype_request = {'persona': 'planner', 'question': '이번 분기 매출이 목표 대비 어떤가?'}
analysis_plan = analysis_branch.invoke(prototype_request)
analysis_plan

PrototypeAnalysisPlan(persona='planner', metric='revenue', dimensions=['month'], comparison='target', chart_hint='grouped_bar')

## 2. read-only PostgreSQL 조회와 DataFrame

In [3]:
sql = (PROJECT_ROOT / 'sql/reference/ts01_planner_target.sql').read_text(encoding='utf-8')
with create_readonly_engine().connect() as connection:
    rows = connection.execute(text(sql)).mappings().all()
df = pd.DataFrame(rows)
assert not df.empty and set(df.columns) == {'month', 'actual_revenue', 'target_revenue', 'achievement_rate'}
df

,month,actual_revenue,target_revenue,achievement_rate
0,2026-07-01,28084673.34,29348483.64,95.69
1,2026-08-01,33988584.17,35518070.46,95.69
2,2026-09-01,24116184.40,25201412.70,95.69


## 3. 결정적 계산, 기본 차트, 근거 요약

In [4]:
actual = float(df['actual_revenue'].sum())
target = float(df['target_revenue'].sum())
achievement = round(actual / target * 100, 2) if target else None
summary = f'이번 분기 완료 주문 매출의 목표 달성률은 {achievement:.2f}%입니다.'
assert '95.69%' in summary

plot_df = df.copy()
plot_df['month'] = plot_df['month'].astype(str)
plot_df[['actual_revenue', 'target_revenue']] = plot_df[['actual_revenue', 'target_revenue']].astype(float)
ax = plot_df.plot(x='month', y=['actual_revenue', 'target_revenue'], kind='bar', figsize=(8, 4))
ax.set_title('Quarterly revenue vs target (Mock)')
ax.set_ylabel('KRW')
plt.tight_layout()
print(summary)
plt.close(ax.figure)

이번 분기 완료 주문 매출의 목표 달성률은 95.69%입니다.


## 4. 선택적 실제 모델 Structured Output smoke test

API 키가 있을 때만 실행한다. 최종 앱은 이 셀의 구현을 복사하지 않고 `src/rolelens`의 모델 팩토리와 분석 체인을 사용한다.

In [5]:
if settings.model_api_key is not None:
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(
        model=settings.model_id,
        api_key=settings.model_api_key.get_secret_value(),
        temperature=0,
        timeout=20,
    ).with_structured_output(PrototypeAnalysisPlan)
    print(model.invoke('planner 관점으로 매출을 목표와 비교하는 계획을 작성해.'))
else:
    print('MODEL_API_KEY/OPENAI_API_KEY가 없어 live model smoke test를 건너뜁니다.')

persona='planner' metric='매출' dimensions=['지역', '제품 카테고리', '판매 채널'] comparison='목표 매출 대비 실적' chart_hint='매출 목표와 실적을 비교하는 바 차트'


## 검증 결론

- `langchain-core`의 `RunnableBranch`만으로 명시적 페르소나 분기가 가능하다.
- 최신 `langchain` 메타패키지는 LangGraph를 전이 의존성으로 가져오므로 MVP에는 포함하지 않는다.
- PostgreSQL 결과는 DataFrame으로 변환하고 숫자는 Python에서 계산한다.
- 다음 Phase에서 공개 스키마, 안전 SQL Tool, 차트 렌더러, InsightService로 추출한다.